# Workflow Analytics & Predictive Automation Demo

This notebook demonstrates the portfolio-safe workflow analysis pipeline using **synthetic event data only**.

It covers:

1. Loading and cleaning event-log data
2. Exploratory workflow analysis
3. Start-event timing analysis
4. Workflow regularity and automation scoring
5. Optional Prophet forecasting
6. Optional LSTM sequence modelling
7. Interpreting the results from a business-process perspective

> **Privacy note:** No original client event logs, credentials, production identifiers, or proprietary data are used in this notebook.


## 1. Setup

The notebook imports the reusable modules from the repository's `src/` folder rather than duplicating the project logic inside the notebook.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

# Locate repository root whether the notebook is run from /notebooks
# or from the repository root.
cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from data_preparation import load_json_events, clean_events, add_dummy_ids
from regularity_analysis import score_automation_candidates

data_path = repo_root / "sample_data" / "synthetic_events.json"

print("Repository root:", repo_root)
print("Synthetic dataset:", data_path)


## 2. Load the Synthetic Event Log

The synthetic dataset mirrors the structure required for the original analytical workflow, including timestamps, event types, workflow IDs, users, work items, and steps.

All identifiers are generated specifically for this public portfolio.


In [ ]:
raw_df = load_json_events(data_path)

print("Raw rows:", len(raw_df))
print("Columns:", list(raw_df.columns))

raw_df.head()


## 3. Clean and Prepare the Data

The ETL stage:

- removes duplicate event IDs;
- keeps relevant `start` and `submit` workflow events;
- converts UTC timestamps to datetime;
- removes invalid placeholder identifiers if present;
- creates a clean DataFrame for analysis.

A second copy is created with sequential dummy identifiers to demonstrate privacy-conscious data handling.


In [ ]:
df = clean_events(raw_df)
df_safe = add_dummy_ids(df)

print("Rows after cleaning:", len(df))
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())

df_safe.head()


## 4. Event-Type Overview

A workflow may contain one Start event, multiple Submit events, and an End event represented in the event history.

The original project found that Submit events can be much more frequent than Start and End events, which is important when selecting data for predictive modelling.


In [ ]:
event_type = df["functionName"].copy()
event_type = event_type.where(df["eventName"].str.lower() != "end", "end")

event_counts = event_type.value_counts()

event_counts


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
event_counts.plot(kind="bar", ax=ax)
ax.set_title("Synthetic Workflow Events by Type")
ax.set_xlabel("Event Type")
ax.set_ylabel("Event Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Workflow Activity by Day of Week

This view helps identify whether operational activity is concentrated on particular days.

For real operational data, this can support workload planning and help reveal recurring workflow behaviour.


In [ ]:
activity = df.copy()
activity["day_name"] = activity["datetime"].dt.day_name()

day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

day_counts = (
    activity["day_name"]
    .value_counts()
    .reindex(day_order, fill_value=0)
)

day_counts


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
day_counts.plot(kind="bar", ax=ax)
ax.set_title("Event Activity by Day of Week")
ax.set_xlabel("Day")
ax.set_ylabel("Event Count")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 6. Start Events by Hour

Start events are analysed separately because they represent workflow initiation behaviour.

Recurring start times can be useful when evaluating whether a workflow is a candidate for scheduled or predictive automation.


In [ ]:
starts = df[df["functionName"] == "start"].copy()
starts["hour"] = starts["datetime"].dt.hour

start_by_hour = starts.groupby("hour").size().reindex(range(24), fill_value=0)

start_by_hour


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(start_by_hour.index, start_by_hour.values, marker="o")
ax.set_title("Start Events by Hour")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Start Event Count")
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()


## 7. Workflow Regularity & Automation Potential

The regularity analysis groups Start events by `flowOriginId` and `userId`.

It considers:

- **Day consistency:** how often events occur on the same weekday
- **Time consistency:** how often events occur around the same 15-minute time window
- **Interval adherence:** how closely the workflow follows an expected daily, weekly, monthly, or longer recurrence

The automation score in this public implementation is a ranking heuristic:

- 60% time consistency
- 20% day consistency
- 20% interval adherence

This is a decision-support measure, not proof that a workflow should be automated.


In [ ]:
automation_candidates = score_automation_candidates(df, min_events=5)

display_columns = [
    "flowOriginId",
    "userId",
    "event_count",
    "interval",
    "day_consistency",
    "time_consistency",
    "interval_adherence",
    "regularity_score",
    "automation_score",
]

automation_candidates[display_columns].head(10)


In [ ]:
top_candidates = automation_candidates.head(10).copy()
top_candidates["candidate"] = (
    top_candidates["flowOriginId"].astype(str)
    + " / "
    + top_candidates["userId"].astype(str)
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_candidates["candidate"], top_candidates["automation_score"])
ax.set_title("Top Synthetic Automation Candidates")
ax.set_xlabel("Automation Score")
ax.set_ylabel("Workflow / User")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 8. Business Interpretation

A high score suggests that historical workflow starts are relatively predictable in terms of timing and recurrence.

Before any real automation decision, an analyst would still need to consider:

- whether the workflow is safe to automate;
- whether human review is required;
- downstream process dependencies;
- exception handling;
- operational and compliance controls;
- business ownership and stakeholder approval;
- monitoring and rollback procedures.


In [ ]:
summary = (
    automation_candidates.groupby("interval")
    .agg(
        candidate_count=("flowOriginId", "size"),
        average_automation_score=("automation_score", "mean"),
    )
    .sort_values("average_automation_score", ascending=False)
)

summary


## 9. Optional: Prophet Forecasting

The repository includes a Prophet-based forecasting module for recurring Start events.

This section is optional because Prophet may not be installed in every notebook environment.

For a workflow/user combination with enough history, the model estimates a future timestamp with strong predicted event intensity.


In [ ]:
try:
    from prophet_forecasting import forecast_by_flow_user

    prophet_results = forecast_by_flow_user(df)
    display(prophet_results.head(10))

except ImportError as exc:
    print("Prophet is not installed in this environment.")
    print("Install project dependencies with: pip install -r requirements.txt")
    print("Details:", exc)

except Exception as exc:
    print("Prophet example could not be completed:", exc)


## 10. Optional: LSTM Sequence Modelling

The LSTM demonstration uses short event sequences to classify whether the next event is a Start event.

Input features include:

- encoded Step ID;
- encoded Work Item / Resource ID;
- time difference between events.

The repository implementation uses a three-event input window and a configurable classification threshold.

The original project used threshold tuning to balance Start and End predictions. In this portfolio example, the default evaluation threshold is `0.6`.


In [ ]:
try:
    from lstm_model import prepare_lstm_dataset, train_lstm

    X, y, metadata = prepare_lstm_dataset(df, window_size=3)

    print("Sequences:", X.shape)
    print("Targets:", y.shape)
    print("Start target share:", float(y.mean()))

    # Keep epochs low for a lightweight portfolio demonstration.
    evaluation = train_lstm(
        X,
        y,
        threshold=0.6,
        epochs=3,
        batch_size=32,
    )

    print("Confusion Matrix:")
    print(evaluation["confusion_matrix"])
    print()
    print("Classification Report:")
    print(evaluation["classification_report"])

except ImportError as exc:
    print("TensorFlow/Keras is not installed in this environment.")
    print("Install project dependencies with: pip install -r requirements.txt")
    print("Details:", exc)

except Exception as exc:
    print("LSTM example could not be completed:", exc)


## 11. Automation Integration Pattern

The repository includes `src/automation_scheduler.py` as a safe architectural example.

It intentionally **does not make real API requests**.

A production implementation would:

1. take an approved workflow candidate;
2. obtain the relevant workflow identifier;
3. retrieve credentials from a secure secret store or environment variable;
4. authenticate against the organisation's documented API;
5. trigger the workflow at an approved time;
6. verify the response;
7. log outcomes without exposing credentials;
8. handle failures and escalation.

This public portfolio repository contains no client endpoints, tokens, API keys, or real workflow identifiers.


In [ ]:
from automation_scheduler import WorkflowClient, WorkflowTriggerConfig

demo_client = WorkflowClient(
    WorkflowTriggerConfig(
        api_base_url="https://example.invalid",
        api_key="demo-placeholder",
    )
)

demo_client.trigger_workflow("demo_workflow_001")


## 12. Key Takeaways

This notebook demonstrates the full analytical story of the portfolio project:

**event logs → ETL → descriptive analysis → recurrence analysis → automation scoring → forecasting / sequence modelling → safe integration design**

From a Business Analyst / Data Analyst perspective, the project demonstrates the ability to connect technical analysis to an operational question: **which recurring workflows are predictable enough to investigate for automation, and how could those predictions be integrated into a controlled business process?**

### Limitations

The public notebook uses synthetic data, so its numerical results should not be compared with the original university project results.

Its purpose is to demonstrate the analytical method, code structure, and business reasoning without publishing confidential client information.
